In [1]:

import numpy as np
from sklearn.metrics import f1_score
from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense


In [2]:
dataset = load_dataset("glue", "sst2")

train_texts = dataset['train']['sentence']
train_labels = dataset['train']['label']

test_texts = dataset['validation']['sentence']
test_labels = dataset['validation']['label']

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [3]:
max_features = 5000
max_len = 50

tokenizer = Tokenizer(num_words=max_features, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

x_train = tokenizer.texts_to_sequences(train_texts)
x_test = tokenizer.texts_to_sequences(test_texts)

x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

y_train = np.array(train_labels)
y_test = np.array(test_labels)

In [4]:
model = Sequential([
    Embedding(max_features, 50, input_length=max_len),
    Conv1D(64, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [5]:
history = model.fit(x_train, y_train, epochs=3, batch_size=64,
                    validation_split=0.2, verbose=1)


Epoch 1/3
842/842 ━━━━━━━━━━━━━━━━━━━━ 14s 15ms/step - accuracy: 0.7198 - loss: 0.5174 - val_accuracy: 0.8676 - val_loss: 0.2966
Epoch 2/3
842/842 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.9003 - loss: 0.2345 - val_accuracy: 0.8894 - val_loss: 0.2572
Epoch 3/3
842/842 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.9233 - loss: 0.1792 - val_accuracy: 0.8984 - val_loss: 0.2507


In [6]:
y_pred_probs = model.predict(x_test)
y_pred = (y_pred_probs > 0.5).astype("int32")
loss, acc = model.evaluate(x_test, y_test, verbose=0)

#f1 score
f1 = f1_score(y_test, y_pred, average='weighted')

#results
print(f"Test Accuracy on SST-2: {acc:.4f}")
print(f"Weighted F1 Score on SST-2: {f1:.4f}")

28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Test Accuracy on SST-2: 0.8165
Weighted F1 Score on SST-2: 0.8161
